In [1]:
import pandas as pd
import os

# ================= 配置区 =================
KG_FILE = "kg.csv"  # 你的 PrimeKG 原文件路径

# 你想侦察的关键词（支持模糊搜索，不区分大小写）
# 我们把它分为几组，方便看结果
SEARCH_GROUPS = {
    "【核心-AD认知】": [
        "Alzheimer", "Dementia", "Cognitive", "Memory"
    ],
    "【外围-高血压】": [
        "Hypertension", "Blood Pressure"
    ],
    "【外围-高血脂】": [
        "Lipid", "Cholesterol", "Hyperlipidemia", "Lipoprotein"
    ],
    "【外围-糖尿病/血糖】": [
        "Diabetes", "Glucose", "Insulin", "Blood sugar"
    ]
}
# =========================================

def explore():
    print(f"🚀 正在加载 {KG_FILE} (可能需要几秒钟)...")
    # 只读取关键列，加快速度
    try:
        df = pd.read_csv(KG_FILE, usecols=['relation', 'x_name', 'x_type', 'y_name', 'y_type'], low_memory=False)
    except ValueError:
        # 如果列名不对，尝试读取所有
        df = pd.read_csv(KG_FILE, low_memory=False)
    
    print(f"✅ 加载完成，共 {len(df)} 条三元组。开始搜索...\n")

    for group_name, keywords in SEARCH_GROUPS.items():
        print(f"================ {group_name} ================")
        # 构建正则表达式: "Alzheimer|Dementia|..."
        pattern = '|'.join(keywords)
        
        # 筛选 x_name 或 y_name 包含关键词的行
        mask = df['x_name'].astype(str).str.contains(pattern, case=False, na=False) | \
               df['y_name'].astype(str).str.contains(pattern, case=False, na=False)
        
        subset = df[mask]
        
        if len(subset) == 0:
            print("  ❌ 未找到任何相关记录。\n")
            continue
            
        print(f"  🔍 命中三元组数量: {len(subset)}")
        
        # 统计都有哪些具体的节点名字（方便你复制到 filter 脚本里）
        # 提取所有相关的名字
        found_names = set(subset[subset['x_name'].astype(str).str.contains(pattern, case=False, na=False)]['x_name']) | \
                      set(subset[subset['y_name'].astype(str).str.contains(pattern, case=False, na=False)]['y_name'])
        
        print(f"  📌 发现的相关节点名 (Top 10):")
        for name in list(found_names)[:10]:
            print(f"     - {name}")
            
        # 展示几条典型的关系，看看质量如何
        print(f"  👀 关系采样 (展示前 3 条):")
        for idx, row in subset.head(3).iterrows():
            print(f"     {row['x_name']} ({row['x_type']}) --[{row['relation']}]--> {row['y_name']} ({row['y_type']})")
        print("\n")

if __name__ == "__main__":
    if os.path.exists(KG_FILE):
        explore()
    else:
        print(f"❌ 找不到文件: {KG_FILE}")

🚀 正在加载 kg.csv (可能需要几秒钟)...
✅ 加载完成，共 8100498 条三元组。开始搜索...

================ 【核心-AD认知】 ================
  🔍 命中三元组数量: 4832
  📌 发现的相关节点名 (Top 10):
     - Semantic dementia
     - Increased proportion of central memory CD4-positive, alpha-beta T cells
     - motor neuron disease with dementia and ophthalmoplegia
     - Focal impaired awareness cognitive seizure with anomia
     - Anterograde memory impairment
     - long-term memory
     - Focal cognitive seizure with dyslexia/alexia
     - amyotrophic lateral sclerosis-parkinsonism-dementia complex
     - Focal aware cognitive seizure with dissociation
     - Focal cognitive seizure with forced thinking
  👀 关系采样 (展示前 3 条):
     Valproic acid (drug) --[contraindication]--> cognitive disorder (disease)
     Lithium citrate (drug) --[contraindication]--> cognitive disorder (disease)
     Lithium carbonate (drug) --[contraindication]--> cognitive disorder (disease)


================ 【外围-高血压】 ================
  🔍 命中三元组数量: 3388
  📌 发现的相关节点名 (To

In [2]:
import pandas as pd

# ================= 配置区 =================
KG_FILE = "kg.csv"  # 你的原始文件，保持不动

# 你关心的核心代谢词 (用于验证是否存在“真身”)
CHECK_KEYWORDS = ["Insulin", "Glucose", "Lipid", "Cholesterol", "Hypertension"]

def main():
    print(f"🚀 正在读取 {KG_FILE} 进行深度体检...")
    try:
        # 只读取关键列，速度快
        df = pd.read_csv(KG_FILE, usecols=['x_name', 'x_type', 'relation', 'y_name', 'y_type'], low_memory=False)
    except:
        df = pd.read_csv(KG_FILE, low_memory=False)
        
    print(f"✅ 加载完毕，共 {len(df)} 条三元组。\n")

    # ================= 1. 验证：代谢关键词的“真身”是否存在？ =================
    print(f"🔎 【验证环节】正在寻找非药物的有效代谢节点...")
    
    # 剔除药物
    mask_no_drug = (df['x_type'] != 'drug') & (df['y_type'] != 'drug')
    df_clean = df[mask_no_drug]
    
    for kw in CHECK_KEYWORDS:
        # 查找包含关键词且非药物的节点
        mask = (df_clean['x_name'].astype(str).str.contains(kw, case=False)) | \
               (df_clean['y_name'].astype(str).str.contains(kw, case=False))
        subset = df_clean[mask]
        
        if len(subset) > 0:
            print(f"   ✅ '{kw}': 发现 {len(subset)} 条非药物关系！")
            # 展示前2个例子，让你放心
            example = subset.iloc[0]
            print(f"      示例: {example['x_name']} ({example['x_type']}) --[{example['relation']}]--> {example['y_name']} ({example['y_type']})")
        else:
            print(f"   ❌ '{kw}': 未找到非药物节点 (可能需要换个同义词)")
            
    # ================= 2. 排雷：寻找干扰性的“万能节点” (Hubs) =================
    print(f"\n💣 【排雷环节】寻找连接数过多的“万能节点” (Top 20)...")
    
    # 统计所有节点的出现次数 (Degree)
    all_nodes = pd.concat([df_clean['x_name'], df_clean['y_name']])
    node_counts = all_nodes.value_counts().head(20)
    
    print("   (如果看到 'Cytoplasm' 或 'Protein binding' 且次数很高，建议在构建时剔除)")
    print("-" * 50)
    print(f"{'节点名称':<40} | {'连接次数':<10}")
    print("-" * 50)
    for node, count in node_counts.items():
        print(f"{node:<40} | {count:<10}")
    
    # ================= 3. 检查：是否存在 'Exposure' 类型 =================
    print(f"\n🧐 【类型检查】当前图谱包含的节点类型分布:")
    types = pd.concat([df_clean['x_type'], df_clean['y_type']]).value_counts()
    print(types)

if __name__ == "__main__":
    main()

🚀 正在读取 kg.csv 进行深度体检...
✅ 加载完毕，共 8100498 条三元组。

🔎 【验证环节】正在寻找非药物的有效代谢节点...
   ✅ 'Insulin': 发现 4058 条非药物关系！
      示例: ACACB (gene/protein) --[phenotype_protein]--> Insulin resistance (effect/phenotype)
   ✅ 'Glucose': 发现 2940 条非药物关系！
      示例: Abnormal glucose homeostasis (effect/phenotype) --[phenotype_phenotype]--> Insulin resistance (effect/phenotype)
   ✅ 'Lipid': 发现 7624 条非药物关系！
      示例: Lipid accumulation in hepatocytes (effect/phenotype) --[phenotype_phenotype]--> Hepatic steatosis (effect/phenotype)
   ✅ 'Cholesterol': 发现 2544 条非药物关系！
      示例: Abnormal circulating lipid concentration (effect/phenotype) --[phenotype_phenotype]--> Abnormal circulating cholesterol concentration (effect/phenotype)
   ✅ 'Hypertension': 发现 1244 条非药物关系！
      示例: ABCA3 (gene/protein) --[phenotype_protein]--> Pulmonary arterial hypertension (effect/phenotype)

💣 【排雷环节】寻找连接数过多的“万能节点” (Top 20)...
   (如果看到 'Cytoplasm' 或 'Protein binding' 且次数很高，建议在构建时剔除)
--------------------------------------------------
节